In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader

import wandb


torch.manual_seed(42)
np.random.seed(42)

In [2]:
database = pd.read_csv('/home/jaume/ConquerX/Mis_apuntes/Pytorch/ejercicio_redes/Pima_Indians_Diabetes.csv')

numero_ceros = (database[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] == 0).sum(axis=0)

database[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] = database[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']].replace(0,np.nan)

columnas = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

# for col in columnas:
#     database[col] = database[col].fillna(database[col].median())

database[columnas] = database[columnas].fillna(database[columnas].median())


database.isna().sum()


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [3]:
database

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0
764,2,122.0,70.0,27.0,125.0,36.8,0.340,27,0
765,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0
766,1,126.0,60.0,29.0,125.0,30.1,0.349,47,1


In [4]:
database['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [5]:
x_datos = database.drop(columns='Outcome')
y_datos = database['Outcome']

print(type(x_datos))
print(type(y_datos))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>


In [6]:
X_train,x_test,y_train,y_test = train_test_split(
    x_datos,y_datos,train_size=0.8,test_size=0.2,random_state=1,stratify=y_datos)

print(type(X_train))
print(type(x_test))
print(type(y_train))
print(type(y_test))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


In [7]:
X_train,X_val,y_train,y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=1,
    stratify=y_train
)

In [8]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [9]:
print(type(X_train))
print(type(X_val))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [10]:
t_x_train = torch.from_numpy(X_train).float().to('cpu')
t_x_val = torch.from_numpy(X_val).float().to('cpu')

t_y_train = torch.from_numpy(y_train.values).float().unsqueeze(1).to('cpu')
t_y_val = torch.from_numpy(y_val.values).float().unsqueeze(1).to('cpu')

print(f"x train: {t_x_train.shape}, x test: {t_x_val.shape} \t y train: {t_y_train.shape}, y test: {t_y_val.shape}")

x train: torch.Size([491, 8]), x test: torch.Size([123, 8]) 	 y train: torch.Size([491, 1]), y test: torch.Size([123, 1])


In [11]:
config = {
    "learning_rate":0.001,
    "epoch":100,
    "batch_size":20,
    "hidden_size":10,
    "dropout":0.3,
    "weight_decay":0.0005
}

In [12]:
train_dataset = TensorDataset(t_x_train,t_y_train)
train_dataset[0]

(tensor([ 0.0601,  0.7273, -1.1673, -0.1306,  0.0060, -0.4110, -0.5636,  0.3504]),
 tensor([0.]))

In [13]:
n_entradas = t_x_train.shape[1]
print(n_entradas)

8


In [14]:
class Red(nn.Module):
    def __init__(self,n_entradas,hiden_size,dropout):
        super().__init__()

        self.capa1 = nn.Linear(n_entradas,hiden_size)
        self.capa2 = nn.Linear(hiden_size,4)
        self.capa3 = nn.Linear(4,1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):

        pred1 = torch.relu(self.capa1(inputs))
        pred_drop = self.dropout(pred1)
        pred2 = torch.relu(self.capa2(pred_drop))
        pred_final = self.capa3(pred2)

        return pred_final

In [15]:
t_y_val.shape

torch.Size([123, 1])

In [16]:
t_x_val.shape

torch.Size([123, 8])

In [17]:
sweep_config = {
    "name":"pima-indians-diabetes",
    "method":"random",
    "metric":{
        "name":"val_accuracy",
        "goal":"maximize"
    },
    "parameters":{
        "learning_rate":{
            "values":[0.01,0.001,0.0001]
        },
        "batch_size":{
            "values":[16,32,64]
        },
        "hidden_size":{
            "values":[6,8,10,12]
        },
        "epoch":{
            "value":100
        },
        "dropout":{
            "value":0.2
        },
        "weight_decay":{
            "value":0.0
        },
    
    }

}

In [ ]:
wandb.login()

In [28]:
def train():

    

    with wandb.init() as run:
        config = run.config
        run.name = f"lr_{config.learning_rate}_bs_{config.batch_size}_hs_{config.hidden_size}"
        
        train_loader = DataLoader(train_dataset,batch_size=config.batch_size,shuffle=True)

        model = Red(
            n_entradas=n_entradas,
            hiden_size=config.hidden_size,
            dropout=config.dropout
        )
        
        criterion = nn.BCEWithLogitsLoss() # Contiene sigmoid

        optimizer = optim.Adam(
            params=model.parameters(),
            lr = config['learning_rate'],
            weight_decay=config.weight_decay
        )

        for epoch in range(1,config['epoch'] +1):

            model.train()
            loss_acumulator = 0

            for x_batch, y_batch in train_loader:

                optimizer.zero_grad()

                y_pred = model(x_batch)
                loss = criterion(y_pred,y_batch)
                loss.backward()
                optimizer.step()
                loss_acumulator += loss.item()
            
            loss_acumulator = loss_acumulator / len(train_loader)

            model.eval()
            with torch.no_grad():
                # VAL
                y_pred_val = model(t_x_val)
                probs = torch.sigmoid(y_pred_val)

                y_class_val = (y_pred_val >= 0).float()
                accuracy_val = ((y_class_val == t_y_val).float().mean()*100).round()

                # TRAIN
                y_pred_train = model(t_x_train)
                y_class_train = (y_pred_train >= 0).float()
                accuracy_train = ((y_class_train == t_y_train).float().mean()*100).round()


            run.log({
                "train_loss":loss_acumulator,
                "train_accuracy":accuracy_train,
                "val_accuracy":accuracy_val,
                
            })


In [24]:
sweep_id = wandb.sweep(sweep=sweep_config,project="Pima-Indians-Diabetes")

KeyboardInterrupt: 

In [30]:
if __name__=='__main__':
    wandb.agent(sweep_id=sweep_id, function=train, count=5)

wandb: Agent Starting Run: ow7aedep with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.2
wandb: 	epoch: 100
wandb: 	hidden_size: 6
wandb: 	learning_rate: 0.0001
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jaume/.netrc.


train_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,████▇▇▇▇▆▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▂▂▂▂▂▂▁▂▁
val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,65
train_loss,0.6587
val_accuracy,65


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: irwkv2vg with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.2
wandb: 	epoch: 100
wandb: 	hidden_size: 8
wandb: 	learning_rate: 0.001
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jaume/.netrc.


train_accuracy,▁▁▁▁▁▇█▇████████████████████████████████
train_loss,███▇▇▆▅▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▂▂▁▁
val_accuracy,▁▁▁▂▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████████████
train_accuracy,80
train_loss,0.39825
val_accuracy,71


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: pam3ghye with config:
wandb: 	batch_size: 16
wandb: 	dropout: 0.2
wandb: 	epoch: 100
wandb: 	hidden_size: 10
wandb: 	learning_rate: 0.001
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jaume/.netrc.


train_accuracy,▁▂▃▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇██▇▇▇▇▇▇▇▇▇█▇█████████
train_loss,█▇▇▆▅▄▄▄▃▃▃▂▃▂▂▂▂▂▂▂▂▁▂▁▂▂▁▁▂▁▂▁▂▁▁▂▁▁▁▁
val_accuracy,▁▅█▇▅█▇█▇▆▆▃▂▂▂▂▃▃▂▃▃▃▅▅▅▅▅▅▅▅▅▅▅▅▃▅▅▅▅▅
train_accuracy,83
train_loss,0.40889
val_accuracy,69


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: ldx8r5yq with config:
wandb: 	batch_size: 16
wandb: 	dropout: 0.2
wandb: 	epoch: 100
wandb: 	hidden_size: 6
wandb: 	learning_rate: 0.001
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jaume/.netrc.


train_accuracy,▁▅▆▇▇▇▇▇████████████████████████████████
train_loss,█▇▆▅▅▄▄▄▃▃▂▂▂▂▂▂▂▂▁▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▅▆▆▆▆▆▆▇▇▇█████████████████████████████
train_accuracy,81
train_loss,0.40642
val_accuracy,72


wandb: Agent Starting Run: nn452s8e with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.2
wandb: 	epoch: 100
wandb: 	hidden_size: 12
wandb: 	learning_rate: 0.01
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jaume/.netrc.


train_accuracy,▁▂▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▅▇▆▆▆▇▆▆▇▇▇▆▇▇▇▇█▇▇█▇▇▇
train_loss,█▆▆▆▅▆▅▆▆▅▄▄▅▅▄▄▃▄▃▅▂▂▂▃▃▃▂▃▃▂▂▂▄▂▂▁▁▁▂▂
val_accuracy,█▇▆█▅▃▃▅▃▄▄▄▃▆▄▃▃▆▄▇▃▅▄▄▅▁▃▅▃▂▄▂▃▄▁▂▄▅▄▃
train_accuracy,87
train_loss,0.33653
val_accuracy,67
